In [ ]:
import os

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
# from langchain.chains import LLMChain
# from langchain.chains import SequentialChain
import os
import json
import pandas as pd
import traceback
from dotenv import load_dotenv
from pypdf import PdfReader
from pydantic import BaseModel, Field
from pypdf import PdfReader

In [ ]:
load_dotenv()

In [ ]:
key=os.getenv("genai_api_key")

In [ ]:
key

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=key,
    temperature=0.5
)

In [ ]:
print(llm)

In [ ]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}


In [ ]:
print(RESPONSE_JSON)

In [ ]:

TEMPLATE="""
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{Response_json}

"""

In [ ]:
from langchain_core.prompts import PromptTemplate


In [ ]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "tone", "Response_json"],
    template=TEMPLATE
)

In [ ]:
print(quiz_generation_prompt)

In [ ]:
quiz_chain = quiz_generation_prompt | llm

In [ ]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [ ]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject", "quiz"], template=TEMPLATE2)

In [ ]:
review_chain= quiz_evaluation_prompt | llm

In [ ]:
from langchain_core.runnables import RunnableLambda

In [ ]:
def generate_quiz(inputs):
    response = quiz_chain.invoke(inputs)
    

    return {
        "subject": inputs["subject"],
        "quiz": response.content
    }


def review_quiz(inputs):
    response = review_chain.invoke({
        "subject": inputs["subject"],
        "quiz": inputs["quiz"]
    })

    return {
        "quiz": inputs["quiz"],
        "review": response.content
    }


generate_evaluate_chain = (
    RunnableLambda(generate_quiz)
    | RunnableLambda(review_quiz)
)

In [ ]:
file_path = r"C:\Users\namde\OneDrive\Desktop\AishuNamdev2006AishuNamdev2006--MCQ-Generator-using-gemini-Langchain-Streamlit\data.txt"

In [ ]:
print(file_path)

In [ ]:
with open(file_path, "r", encoding="utf-8") as file:
    TEXT = file.read()

print(TEXT)

In [ ]:
json.dumps(RESPONSE_JSON)

In [ ]:
NUMBER=5 
SUBJECT="genrative ai"
TONE="simple"

In [ ]:
result = generate_evaluate_chain.invoke({
    "text": TEXT,
    "number": NUMBER,
    "subject": SUBJECT,
    "tone": TONE,
    "Response_json": json.dumps(RESPONSE_JSON)
})
print(result["quiz"])
print(result["review"])